# Il pattern di cio che manca riconosce il paziente

**CAF GIAS, Universita degli Studi Link, modulo 2.4** | Michele Ferramola

Dimostrazione per il blocco *Gestione dei missing data*, in due parti:
il pattern di cio che manca (5 minuti) e le tre strategie all'opera (3 minuti).

---

### La domanda

Un dataset clinico ha migliaia di colonne. Alcune sono piene, altre hanno dei buchi.
Se **cancello tutti i valori** e tengo solo l'informazione *c'e il dato oppure no*,
quanto riesco ancora a sapere del paziente?

### Il dato

NHANES 2013-2014, indagine pubblica statunitense su salute e nutrizione.
6.113 adulti, 1.810 variabili.
Il file che carichiamo **non contiene nessun valore clinico**: solo `True` e `False`,
piu l'etichetta del sesso che useremo come bersaglio.


In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

URL = "https://raw.githubusercontent.com/MicheleFerramola00/lezioni/caf-unilink-m2.4/dati/mancanza_nhanes_adulti.parquet"

M = pd.read_parquet(URL)
y = M.pop("sesso_F").values          # 1 = donna, 0 = uomo

print(f"pazienti: {len(M)}   colonne: {M.shape[1]}   donne: {y.mean():.1%}")
print(f"tipi di dato presenti nel file: {set(map(str, M.dtypes))}")
M.iloc[:6, :6]


Sopra non si legge nessun esito clinico. Solo `True` dove il dato manca e `False` dove c'e.

Teniamo le colonne che hanno un minimo di variabilita, cioe che non sono ne sempre piene ne sempre vuote.


In [ ]:
M = M.loc[:, (M.mean() > 0.01) & (M.mean() < 0.99)]
print(f"colonne con variabilita: {M.shape[1]}")

tassoF, tassoM = M[y == 1].mean(), M[y == 0].mean()
divario = (tassoF - tassoM)

top = pd.concat([divario.nsmallest(10), divario.nlargest(10)]).sort_values()
ax = top.plot.barh(figsize=(9, 6), color=["#155999" if v < 0 else "#D4A13A" for v in top])
ax.set_xlabel("quanto piu spesso il dato manca   (blu = manca agli uomini, oro = manca alle donne)")
ax.set_title("Le venti colonne in cui il buco distingue di piu i due sessi")
ax.axvline(0, color="black", lw=0.8); plt.tight_layout(); plt.show()


Nessuna sorpresa: salute riproduttiva, HPV, comportamento sessuale.
Domande che a un sesso non vengono proprio rivolte.

### Adesso la prova

Alleno una regressione logistica **sulla sola matrice dei buchi** e le chiedo di indovinare il sesso.


In [ ]:
def prova(X, etichetta):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    mod = LogisticRegression(max_iter=3000).fit(Xtr, ytr)
    auroc = roc_auc_score(yte, mod.predict_proba(Xte)[:, 1])
    print(f"{etichetta:<46} AUROC = {auroc:.3f}   ({X.shape[1]} colonne)")
    return auroc

prova(M, "tutte le colonne di mancanza")


### L'obiezione, che e legittima

*"Certo: gravidanza, prostata, HPV. Sono domande che si fanno a un sesso solo.*
*Togli quelle e il problema sparisce."*

Proviamo. Rimuovo **ogni colonna appartenente a un dominio sesso-specifico**:
salute riproduttiva, HPV, comportamento sessuale, prostata, gravidanza, ciclo mestruale.


In [ ]:
prefissi = ("rh_", "vag_", "sx_", "imm_hpv")
parole   = ("prostat", "pregn", "menstr", "ovar", "hysterect")
sesso_specifiche = [c for c in M.columns
                    if c.lower().startswith(prefissi) or any(p in c.lower() for p in parole)]
print(f"colonne rimosse: {len(sesso_specifiche)}")
print()

prova(M, "tutte le colonne")
prova(M.drop(columns=sesso_specifiche), "senza i domini sesso-specifici")


Il modello continua a riconoscere il sesso quasi perfettamente.

### Fino a che punto regge

Alzo il livello: elimino **ogni colonna** il cui tasso di mancanza differisce fra i due sessi
piu di una certa soglia. E un filtro che nella pratica nessuno applicherebbe mai.


In [ ]:
scala = []
for soglia in (1.00, 0.30, 0.10, 0.05, 0.02):
    tenute = M.loc[:, divario.abs() <= soglia]
    scala.append((f"divario <= {soglia:.0%}", prova(tenute, f"divario di mancanza <= {soglia:.0%}")))

print()
lab = [s for s, _ in scala]; val = [v for _, v in scala]
ax = plt.figure(figsize=(8, 4)).gca()
ax.bar(lab, val, color="#103F6D"); ax.axhline(0.5, color="#D4A13A", ls="--", label="caso")
ax.set_ylim(0.4, 1.02); ax.set_ylabel("AUROC nel riconoscere il sesso")
ax.set_title("Quanto devo mutilare il dato prima che il segnale sparisca")
ax.legend(); plt.tight_layout(); plt.show()


### Dove vive davvero il segnale

Ultima verifica: e colpa delle macchine di laboratorio o delle domande del questionario?


In [ ]:
lab_fis = [c for c in M.columns if c.split("_")[0] in
           ("lab", "bp", "bm", "bmx", "chol", "gluc", "cbc", "hdl", "ldl",
            "tri", "ins", "alb", "cre", "urx", "phy", "grip", "dxa", "ohx")]
prova(M[lab_fis], "solo laboratorio ed esame fisico")


---

## Cosa abbiamo visto

1. **Il buco non e assenza di informazione.** La sola mappa di cio che manca identifica il sesso.
2. **Togliere le variabili ovvie non basta.** Il segnale e distribuito su centinaia di colonne,
   ciascuna delle quali da sola sembra innocua.
3. **Il segnale vive nel processo, non nello strumento.** Le colonne di laboratorio quasi non lo portano:
   lo porta cio che e stato chiesto, a chi, e quando.

### Perche riguarda l'imaging

In imaging il buco non e una cella vuota: e **un paziente intero che non compare nel dataset**,
perche nessuno gli ha prescritto l'esame. La prescrizione e una decisione umana, con determinanti noti:
accesso, lingua, eta, presunzione diagnostica.
Lo stesso meccanismo, con una traccia ancora piu difficile da vedere.

### La prima verifica da portare in gara

> **Chi manca dal vostro dataset, e per quale meccanismo di selezione?**

Ancoraggio: AI Act art. 10 (governance dei dati), STANDING Together (documentazione della diversita dei dataset).


---

## Parte 2 · Le tre strategie all'opera

La prima parte ha mostrato che il buco porta informazione. Adesso la domanda pratica: **come si gestisce**.

Per sapere se una strategia regge bisogna conoscere il valore vero. Prendo quindi una misura che NHANES
ha per quasi tutti, la **pressione sistolica**, e la tolgo io a circa il 40% dei pazienti, in tre modi diversi:
gli stessi tre modi di mancare delle slide.

In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression

URL_VALORI = "https://raw.githubusercontent.com/MicheleFerramola00/lezioni/caf-unilink-m2.4/dati/valori_nhanes_adulti.parquet"
V = pd.read_parquet(URL_VALORI)
p = V.dropna(subset=["pas"])
vero = p.pas.mean()
print(f"adulti con la pressione misurata: {len(p)}   pressione sistolica media, quella vera: {vero:.1f} mmHg")

### Strategie 1 e 2: tenere solo chi ha tutto, oppure stimare il valore che manca

- **MCAR**: manca per caso, al 40% dei pazienti.
- **MAR**: manca al 70% dei giovani (sotto i 40 anni) e al 25% degli altri. L'eta e registrata.
- **MNAR**: manca al 70% di chi ha la pressione alta (130 o piu) e al 25% degli altri.

Per ciascun modo, di quanto sbagliano la media le due strategie: tenere solo chi ha il dato,
oppure stimare il dato mancante a partire da eta e sesso.

In [ ]:
rng = np.random.default_rng(0)
regole = {"MCAR, manca per caso": np.full(len(p), 0.40),
          "MAR, manca ai giovani": np.where(p.eta < 40, 0.70, 0.25),
          "MNAR, manca a chi ha la pressione alta": np.where(p.pas >= 130, 0.70, 0.25)}
righe = []
for nome, prob in regole.items():
    manca = rng.random(len(p)) < prob
    X = p[["eta", "donna"]]
    stima = LinearRegression().fit(X[~manca], p.pas[~manca]).predict(X)
    righe.append((nome, f"{manca.mean():.0%}", p.pas[~manca].mean() - vero, np.where(manca, stima, p.pas).mean() - vero))
pd.DataFrame(righe, columns=["come manca", "quanti mancano", "errore dei casi completi (mmHg)",
                             "errore dell'imputazione con eta e sesso (mmHg)"]).round(1)

- **Tenere solo chi ha tutto** regge se il dato manca per caso. Se manca ai giovani, la media sale.
- **Stimare il valore che manca** corregge quando il motivo della mancanza e registrato: qui l'eta.
- **Se manca proprio a chi ha la pressione alta** sbagliano tutte e due: il dato che servirebbe
  e quello che manca, e nessun calcolo lo recupera dai soli dati disponibili.

### Strategia 3: usare la mancanza come informazione

Due ospedali, pazienti simili. Nell'ospedale **A** la glicemia si chiede a chi e sospetto di diabete;
nell'ospedale **B** si chiede a tutti sopra i 50 anni. Un modello impara in A a riconoscere il diabete
da eta, BMI e glicemia, e poi viene usato in B.

In [ ]:
q = V.dropna(subset=["glicemia", "bmi"]).copy()
q["diabete"] = ((q.diabete_dichiarato == 1) | (q.hba1c >= 6.5)).astype(int)
rng = np.random.default_rng(0)
in_A = rng.random(len(q)) < 0.5
A, B = q[in_A].copy(), q[~in_A].copy()
A["misurata"] = rng.random(len(A)) < np.where(A.diabete == 1, 0.85, 0.30)   # A: si chiede a chi e sospetto
B["misurata"] = rng.random(len(B)) < np.where(B.eta >= 50, 0.90, 0.10)      # B: si chiede a tutti sopra i 50 anni
allena = rng.random(len(A)) < 0.7
Atr, Ate = A[allena], A[~allena]

mediana = Atr.glicemia[Atr.misurata].median()
stima = LinearRegression().fit(Atr.loc[Atr.misurata, ["eta", "bmi"]], Atr.glicemia[Atr.misurata])

def colonne(x, indicatore):
    g = x.glicemia.where(x.misurata)
    X = pd.DataFrame({"eta": x.eta, "bmi": x.bmi})
    if indicatore:      # strategia 3: il buco diventa una variabile
        X["glicemia"], X["manca"] = g.fillna(mediana), (~x.misurata).astype(int)
    else:               # strategia 2: il buco si riempie con una stima da eta e BMI
        X["glicemia"] = g.fillna(pd.Series(stima.predict(x[["eta", "bmi"]]), index=x.index))
    return X

righe = []
for nome, indicatore in (("usa la mancanza come informazione", True), ("stima il valore che manca", False)):
    mod = LogisticRegression(max_iter=2000).fit(colonne(Atr, indicatore), Atr.diabete)
    righe.append((nome, roc_auc_score(Ate.diabete, mod.predict_proba(colonne(Ate, indicatore))[:, 1]),
                  roc_auc_score(B.diabete, mod.predict_proba(colonne(B, indicatore))[:, 1])))
pd.DataFrame(righe, columns=["il modello", "AUROC nell'ospedale A", "AUROC nell'ospedale B"]).round(3)

In A il buco aiuta, perche vuol dire *non era sospetto*. In B lo stesso buco vuol dire *ha meno di 50 anni*,
e il modello che ci si era appoggiato perde. Il modello che stima il valore mancante rende meno in A, ma regge in B.

---

## Cosa abbiamo visto nella parte 2

1. **Tenere solo chi ha tutto** regge solo se il dato manca per caso.
2. **Stimare il valore che manca** regge se il motivo della mancanza e registrato e la stima lo usa.
3. **Usare la mancanza come informazione** regge finche il processo che crea i buchi resta lo stesso.
4. **Se manca proprio a chi sta peggio**, nessuna strategia basta: servono un'analisi di sensibilita,
   il recupero del dato su un campione di pazienti, la registrazione della causa.

### Da verificare in gara

> **Sotto quale ipotesi sono stati gestiti i dati mancanti? E se il modello usa la mancanza come informazione, in quali ospedali e stato verificato?**